<a href="https://colab.research.google.com/github/shoh0806/Deep-Learning-Project/blob/main/ViT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. 라이브러리 설치

In [ ]:
!pip install transformers -q

2. 라이브러리 import

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import ViTModel, ViTImageProcessor
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score
from PIL import Image
import warnings
warnings.filterwarnings('ignore')
import random

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True  # 이건 GPU 연산을 항상 같은 순서로 실행하게 하는 코드입니다 없으면 gpu에서 랜덤성이 생길 수 있다고 합니다.
    torch.backends.cudnn.benchmark = False  #GPU 최적화 알고리즘 고정 , 없으면 실행할 때마다 다른 알고리즘 선택할 수 있음

set_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"사용 장치: {device}")

 3. Drive 연결 및 데이터 로드

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

BASE_DIR = '/content/drive/MyDrive/clickbait_project'
CSV_PATH = f'{BASE_DIR}/dataset.csv'
IMG_DIR  = f'{BASE_DIR}/thumbnails'

df = pd.read_csv(CSV_PATH)
print(f"전체 데이터: {len(df)}개")
print(f"클릭베이트(1): {len(df[df['label']==1])}개")
print(f"정상(0): {len(df[df['label']==0])}개")

# 4. 데이터 분할

In [ ]:
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['label'])

print(f"train: {len(train_df)}개")
print(f"val:   {len(val_df)}개")
print(f"test:  {len(test_df)}개")

# 5. Feature Extractor 및 Dataset 클래스

In [ ]:
extractor = ViTImageProcessor.from_pretrained('google/vit-base-patch16-224')

class ImageDataset(Dataset):
    def __init__(self, df, extractor, img_dir):
        self.video_ids = df['video_id'].tolist()
        self.labels    = df['label'].tolist()
        self.extractor = extractor
        self.img_dir   = img_dir

    def __len__(self):
        return len(self.video_ids)

    def __getitem__(self, idx):
        img_path = f"{self.img_dir}/{self.video_ids[idx]}.jpg"
        image = Image.open(img_path).convert('RGB')
        inputs = self.extractor(images=image, return_tensors='pt')
        return {
            'pixel_values': inputs['pixel_values'].squeeze(),
            'label':        torch.tensor(self.labels[idx], dtype=torch.long)
        }

train_dataset = ImageDataset(train_df, extractor, IMG_DIR)
val_dataset   = ImageDataset(val_df, extractor, IMG_DIR)
test_dataset  = ImageDataset(test_df, extractor, IMG_DIR)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=32)
test_loader  = DataLoader(test_dataset, batch_size=32)

print("Dataset 준비 완료!")
print(f"train loader: {len(train_loader)}배치")

#  6. ViT 모델 정의

In [ ]:
class ViTClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.vit = ViTModel.from_pretrained('google/vit-base-patch16-224')
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(768, 2)

    def forward(self, pixel_values):
        outputs = self.vit(pixel_values=pixel_values)
        cls_vector = outputs.last_hidden_state[:, 0, :]
        cls_vector = self.dropout(cls_vector)
        return self.classifier(cls_vector)

model = ViTClassifier().to(device)
print("ViT 모델 준비 완료!")
print(f"파라미터 수: {sum(p.numel() for p in model.parameters()):,}")

# 7. 학습 함수

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-5)
criterion = nn.CrossEntropyLoss()

def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, preds, targets = 0, [], []

    for batch in loader:
        pixel_values = batch['pixel_values'].to(device)
        labels       = batch['label'].to(device)

        optimizer.zero_grad()
        outputs = model(pixel_values)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds   += outputs.argmax(1).cpu().tolist()
        targets += labels.cpu().tolist()

    return total_loss/len(loader), f1_score(targets, preds, average='macro'), accuracy_score(targets, preds)

def eval_epoch(model, loader, criterion):
    model.eval()
    total_loss, preds, targets = 0, [], []

    with torch.no_grad():
        for batch in loader:
            pixel_values = batch['pixel_values'].to(device)
            labels       = batch['label'].to(device)

            outputs    = model(pixel_values)
            loss       = criterion(outputs, labels)

            total_loss += loss.item()
            preds   += outputs.argmax(1).cpu().tolist()
            targets += labels.cpu().tolist()

    return total_loss/len(loader), f1_score(targets, preds, average='macro'), accuracy_score(targets, preds)

print("학습 함수 준비 완료!")

#  8. 학습 실행

In [ ]:
EPOCHS = 10
best_val_f1 = 0
patience_counter = 0
PATIENCE = 3

for epoch in range(EPOCHS):
    train_loss, train_f1, train_acc = train_epoch(model, train_loader, optimizer, criterion)
    val_loss, val_f1, val_acc       = eval_epoch(model, val_loader, criterion)

    print(f"Epoch {epoch+1}/{EPOCHS}")
    print(f"  train loss:{train_loss:.4f} f1:{train_f1:.4f} acc:{train_acc:.4f}")
    print(f"  val   loss:{val_loss:.4f} f1:{val_f1:.4f} acc:{val_acc:.4f}")

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        patience_counter = 0
        torch.save(model.state_dict(), f'{BASE_DIR}/vit_best.pt')
        print(f"  → 모델 저장!")
    else:
        patience_counter += 1
        print(f"  → patience {patience_counter}/{PATIENCE}")
        if patience_counter >= PATIENCE:
            print(f"\nEarly Stopping! 최고 val f1: {best_val_f1:.4f}")
            break

# 9. 테스트 평가

In [ ]:
model.load_state_dict(torch.load(f'{BASE_DIR}/vit_best.pt'))
test_loss, test_f1, test_acc = eval_epoch(model, test_loader, criterion)

print("========== ViT 최종 결과 ==========")
print(f"test F1:  {test_f1:.4f}")
print(f"test Acc: {test_acc:.4f}")

vit_result = {'f1': test_f1, 'acc': test_acc}